There are 4 ways to connect ADLS Gen 2 with Databricks :
1. Mounting using OAuth
2. Using ASP 
3. Using SAS token
4. Using account Key

### Mount ADLS using OAuth 
**Databricks no logger recommends mounting external storage to DBFS**

https://learn.microsoft.com/en-us/azure/databricks/storage/azure-storage

1. application-id - Also known as Client ID, 
2. scope-name - This is databricks secret scope name.
3. service-credential-key-name - This is the key containing the client secret
4. directory-id - Also known as Tenand ID, This is the Azure Active Directory application. 
5. storage_account_name - Name of the storage account
6. container_name - Name of the container of the storage account
7. mount_name - the mount name to be given for the container.

In [0]:
container_name = 'raw'
storage_account_name = 'datalake_name'
mount_name = 'mount_name'

In [0]:
configs = {"fs.azure.account.auth.type": "OAuth",
          "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
          "fs.azure.account.oauth2.client.id": "<application-id>",
          "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope="<scope-name>",key="<service-credential-key-name>"),
          "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/<directory-id>/oauth2/token"}

# Optionally, you can add <directory-name> to the source URI of your mount point.
dbutils.fs.mount(
  source = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/",
  mount_point = f"/mnt/{mount_name}",
  extra_configs = configs)

#### To unmount a moint point

In [0]:
dbutils.fs.unmount(f'/mnt/{mount_name}')

In [0]:
dbutils.fs.refreshMounts()

### Access ADLS Gen2 or Blob Storage using OAuth 2.0 with Azure Service principal

OAuth 2.0 - it gives limited access to a user to only access specific resources 

In [0]:
def ADLSconnectionSP(storage_account_name, application_id, service_credential, tenant_id):
    spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", application_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", service_credential)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f'https://login.microsoftonline.com/{tenant_id}/oauth2/token')

### Access ADLS Gen2 or Blob Storage using SAS token

### Access ADLS Gen2 or Blob Storage using account key 

In [0]:
def ADLSconnectionAccountKey(storage_account_name, account_key):
    spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", dbutils.secrets.get(scope = "databricks-secret-scope", key = account_key))


In [0]:
spark.conf.get("spark.databricks.cloudProvider")

In [0]:
spark.conf.get("eventLog.rolloverIntervalSeconds")